# DPO Training mit Hugging Face DPOTrainer (trl)
### Masterarbeit: Automatische Übersetzung von Alltagssprache (AS) in Leichte Sprache (LS)

Dieses Notebook implementiert DPO mithilfe der offiziellen `trl` Bibliothek (`DPOTrainer`) speziell konfiguriert für Seq2Seq (Encoder-Decoder) Architekturen.

Der Ablauf ist wie folgt:
1. **Preference-Dataset Erstellung/Laden:** Generieren oder Laden der Preference-Daten (chosen/rejected).
2. **DPO Training mit DPOTrainer:** Wir laden das Modell mit LoRA/PEFT und trainieren es über den `DPOTrainer` von Hugging Face.

In [1]:
import os
import sys

# Arbeitsverzeichnis auf das Root-Verzeichnis setzen
while not os.path.exists(".git"):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir("..")

print("Aktuelles Arbeitsverzeichnis:", os.getcwd())

Aktuelles Arbeitsverzeichnis: /home/fiete/master-thesis


In [2]:
# Bibliotheken installieren/upgraden falls nötig
# !pip install trl peft transformers datasets accelerate -U

In [3]:
import json
import glob
import random
import copy
import gc
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
from transformers import BertTokenizerFast, BertModel
from datasets import Dataset as HFDataset
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig, get_peft_model, TaskType

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Nutze Device: {DEVICE}")

Nutze Device: cuda


In [4]:
# ==============================================================================
# KONFIGURATION
# ==============================================================================
CORPUS_DIR = "data/corpus/4_normalized_clean"
OUTPUT_DIR = "results/models/seq2seq_dpo_hf_trainer"
PREFERENCE_DATA_PATH = "results/dpo_preferences_winner_loser.json"
SYNTHETIC_MODEL_PATH = "results/models/gbert_synthetic_regression.pt"
SFT_MODEL_PATH = "results/models/best_sft_w05_w05_gbert_temp.pt"
MODEL_NAME_GBERT = "deepset/gbert-base"
MODEL_NAME = "facebook/mbart-large-50"
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 256

W_STYLE = 0.5
W_SEM = 0.5
BETA = 0.1 # DPO Temperatur-Parameter

os.makedirs(os.path.dirname(PREFERENCE_DATA_PATH), exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. GBERT Regressor & SBERT (Reward-Berechnung)
Wir definieren den GBERT Regressor für Style-Scores und laden SBERT für semantische Ähnlichkeiten.

In [5]:
class GBERTRegressor(nn.Module):
    def __init__(self, model_name=MODEL_NAME_GBERT, dropout_rate=0.1):
        super(GBERTRegressor, self).__init__()
        self.gbert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout_rate)
        self.regressor = nn.Linear(self.gbert.config.hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, input_ids, attention_mask):
        outputs = self.gbert(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = outputs.last_hidden_state[:, 0, :]
        dropped = self.dropout(cls_rep)
        logits = self.regressor(dropped)
        return self.sigmoid(logits)

print("Lade GBERT & SBERT für Reward-Berechnung...")
gbert_tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME_GBERT)
gbert_regressor = GBERTRegressor().to(DEVICE)

# Robustes Laden des State Dicts (umbenennen von gbert -> bert oder umgekehrt falls nötig)
state_dict = torch.load(SYNTHETIC_MODEL_PATH, map_location=DEVICE)
new_state_dict = {}
for key, val in state_dict.items():
    if key.startswith("bert."):
        new_key = key.replace("bert.", "gbert.")
    else:
        new_key = key
    new_state_dict[new_key] = val

gbert_regressor.load_state_dict(new_state_dict)
gbert_regressor.eval()

sbert_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=str(DEVICE))
print("Reward-Modelle erfolgreich geladen!")

Lade GBERT & SBERT für Reward-Berechnung...
Reward-Modelle erfolgreich geladen!


In [6]:
def predict_simplicity_score(texts):
    inputs = gbert_tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        preds = gbert_regressor(inputs["input_ids"], inputs["attention_mask"])
    return preds.cpu().squeeze(-1).numpy().tolist()

def predict_semantic_similarity(source_texts, generated_texts):
    src_emb = sbert_model.encode(source_texts, convert_to_tensor=True)
    gen_emb = sbert_model.encode(generated_texts, convert_to_tensor=True)
    cos_sims = util.cos_sim(src_emb, gen_emb)
    # Wir wollen nur die Diagonale (Matching Paare)
    return cos_sims.diag().cpu().numpy().tolist()

class CompositeRewardEvaluator:
    def __init__(self, w_style=0.5, w_sem=0.5):
        self.w_style = w_style
        self.w_sem = w_sem
        
    def compute_reward(self, source_texts, generated_texts):
        # Style-Score: Simplicity Predicter (höher = besser)
        style_scores = predict_simplicity_score(generated_texts)
        # Semantische Ähnlichkeit zwischen AS und LS
        sem_scores = predict_semantic_similarity(source_texts, generated_texts)
        
        rewards = []
        for style, sem in zip(style_scores, sem_scores):
            # Kombinierter Reward
            reward = self.w_style * style + self.w_sem * sem
            rewards.append(reward)
        return rewards, style_scores, sem_scores

reward_evaluator = CompositeRewardEvaluator(w_style=W_STYLE, w_sem=W_SEM)

## 2. Laden des trainierten SFT Modells & Datenladen
Wir laden das zuvor in Phase 1 trainierte SFT-Modell.

In [7]:
print(f"Lade Basismodell: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

if "mbart" in MODEL_NAME.lower():
    tokenizer.src_lang = "de_DE"
    tokenizer.tgt_lang = "de_DE"

# Laden der trainierten SFT Gewichte aus Phase 1
if os.path.exists(SFT_MODEL_PATH):
    print(f"Lade trainierte SFT-Gewichte aus: {SFT_MODEL_PATH}...")
    model.load_state_dict(torch.load(SFT_MODEL_PATH, map_location=DEVICE))
    print("SFT Gewichte erfolgreich geladen!")
else:
    print(f"[WARNING] Keine SFT Gewichte unter {SFT_MODEL_PATH} gefunden. Es wird das untrainierte Basismodell verwendet.")

Lade Basismodell: facebook/mbart-large-50...
Lade trainierte SFT-Gewichte aus: results/models/best_sft_w05_w05_gbert_temp.pt...
SFT Gewichte erfolgreich geladen!


In [8]:
# Lade Rohdaten
def load_corpus_pairs(corpus_dir=CORPUS_DIR):
    pairs = []
    json_files = glob.glob(os.path.join(corpus_dir, "*.json"))
    for file_path in json_files:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            for item in data.get("pairs", []):
                as_text = item.get("as_text", "").strip()
                ls_text = item.get("ls_text", "").strip()
                if as_text and ls_text:
                    pairs.append({"as_text": as_text, "ls_text": ls_text})
    return pairs

raw_pairs = load_corpus_pairs()
print(f"Geladene Paare: {len(raw_pairs)}")

Geladene Paare: 1466


## 3. Preference-Dataset vorbereiten
Verwende **entweder** Option A (Neu generieren) **oder** Option B (Vortrainierte JSON-Daten von Festplatte laden).

### OPTION A: Preference Dataset neu generieren
Führe diese Zelle aus, wenn du die Übersetzungen neu generieren und bewerten möchtest. Das Ergebnis wird unter `results/dpo_preferences_winner_loser.json` gespeichert.

In [9]:
# def generate_alternative_translation(model, tokenizer, as_texts):
#     prompts = ["Übersetze in Leichte Sprache: " + t for t in as_texts]
#     inputs = tokenizer(prompts, padding=True, truncation=True, max_length=MAX_SOURCE_LEN, return_tensors="pt").to(DEVICE)
#     with torch.no_grad():
#         outputs = model.generate(
#             input_ids=inputs["input_ids"],
#             attention_mask=inputs["attention_mask"],
#             max_length=MAX_TARGET_LEN,
#             num_beams=4,
#             early_stopping=True
#         )
#     return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# print("Generiere Preference Dataset und bewerte Kandidaten...")
# preference_data = []

# # Batch-Verarbeitung zur Beschleunigung
# BATCH_SIZE = 8
# for i in tqdm(range(0, len(raw_pairs), BATCH_SIZE)):
#     batch = raw_pairs[i : i + BATCH_SIZE]
#     as_texts = [item["as_text"] for item in batch]
#     ls_ground_truths = [item["ls_text"] for item in batch]
    
#     # Generiere alternative Übersetzung mit dem SFT-Modell
#     generated_translations = generate_alternative_translation(model, tokenizer, as_texts)
    
#     # Berechne Rewards für Ground Truth (GT) und die Generierung (GEN)
#     rewards_gt, style_gt, sem_gt = reward_evaluator.compute_reward(as_texts, ls_ground_truths)
#     rewards_gen, style_gen, sem_gen = reward_evaluator.compute_reward(as_texts, generated_translations)
    
#     for j in range(len(batch)):
#         as_txt = as_texts[j]
#         gt_txt = ls_ground_truths[j]
#         gen_txt = generated_translations[j]
        
#         # Vergleiche Rewards: Höherer Reward gewinnt (chosen), niedrigerer verliert (rejected)
#         if rewards_gt[j] > rewards_gen[j]:
#             chosen = gt_txt
#             rejected = gen_txt
#             chosen_r, rejected_r = rewards_gt[j], rewards_gen[j]
#         elif rewards_gen[j] > rewards_gt[j]:
#             chosen = gen_txt
#             rejected = gt_txt
#             chosen_r, rejected_r = rewards_gen[j], rewards_gt[j]
#         else:
#             continue
            
#         preference_data.append({
#             "prompt": "Übersetze in Leichte Sprache: " + as_txt,
#             "chosen": chosen,
#             "rejected": rejected,
#             "as_text_raw": as_txt,
#             "chosen_reward": float(chosen_r),
#             "rejected_reward": float(rejected_r)
#         })

# print(f"Preference Dataset fertiggestellt: {len(preference_data)} Paare.")

# # Export der generierten Daten
# with open(PREFERENCE_DATA_PATH, "w", encoding="utf-8") as f:
#     json.dump(preference_data, f, ensure_ascii=False, indent=4)
# print(f"Winner-Loser-Preference-Dataset neu generiert und gespeichert unter: {PREFERENCE_DATA_PATH}")

### OPTION B: Preference Dataset aus Cache laden
Führe diese Zelle aus, um das DPO-Training sofort auf bereits generierten Daten zu starten (Generierungszeit wird übersprungen).

In [10]:
if os.path.exists(PREFERENCE_DATA_PATH):
    print(f"Lade Datensatz von Festplatte: {PREFERENCE_DATA_PATH}...")
    with open(PREFERENCE_DATA_PATH, "r", encoding="utf-8") as f:
        preference_data = json.load(f)
    print(f"Erfolgreich {len(preference_data)} Preference-Paare geladen!")
else:
    print(f"[ERROR] Keine Datei unter {PREFERENCE_DATA_PATH} gefunden. Bitte führe zuerst Option A aus.")

Lade Datensatz von Festplatte: results/dpo_preferences_winner_loser.json...
Erfolgreich 1460 Preference-Paare geladen!


## 4. Übersicht der Winner-Loser-Daten
Wir betrachten eine Vorschau des Datensatzes vor dem Training.

In [11]:
# Zeige die ersten 3 Beispiele an
df_pref = pd.DataFrame(preference_data)
df_pref[["as_text_raw", "chosen", "rejected", "chosen_reward", "rejected_reward"]].head(3)

,as_text_raw,chosen,rejected,chosen_reward,rejected_reward
0,Was ist die Ursache für Weitsichtigkeit? Sympt...,Was ist Weitsichtigkeit? Weitsichtigkeit ist e...,Was ist Weitsichtigkeit? Weitsichtigkeit ist e...,0.857738,0.826514
1,Was ist Baldrian? Gegen was hilft Baldrian? Gi...,Was ist Baldrian? Baldrian ist eine Heilpflanz...,Was ist Baldrian? Baldrian ist eine Heilpflanz...,0.929794,0.888209
2,Was ist eine Magenschleimhautentzündung? Wie m...,Was ist eine Magenschleimhaut-Entzündung? Der ...,Was ist eine Magenschleimhaut-Entzündung? Der ...,0.815273,0.802488


## 5. DPO Training mit dem Hugging Face DPOTrainer und LoRA
Wir konfigurieren nun LoRA und den `DPOTrainer` für das Seq2Seq Modell.

In [12]:
# Dataset für Hugging Face konvertieren (Nur Textspalten übergeben, damit TRL das Seq2Seq-Preprocessing selbst anstösst)
df_trl = df_pref[["prompt", "chosen", "rejected"]]
hf_dataset = HFDataset.from_pandas(df_trl)

# Train/Val-Split
hf_dataset = hf_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = hf_dataset["train"]
eval_dataset = hf_dataset["test"]
print(f"Train-Größe: {len(train_dataset)} | Val-Größe: {len(eval_dataset)}")

Train-Größe: 1241 | Val-Größe: 219


In [13]:
# LoRA Konfiguration für MBart
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

# DPO Konfiguration über DPOConfig
# Da wir ein Seq2Seq-Modell nutzen, ist es entscheidend, dass max_length, 
# sowie das Model-Attribut is_encoder_decoder korrekt geladen werden.
# Da is_encoder_decoder nicht in DPOConfig existiert, liest TRL dies aus model.config aus.
training_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,          # VRAM-Schonung
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,          # Effektive Batch-Größe = 16
    learning_rate=5e-5,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=10,
    max_steps=200,
    warmup_steps=20,
    save_strategy="steps",
    save_steps=100,
    bf16=True if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else False,
    fp16=True if not torch.cuda.is_bf16_supported() else False,
    report_to="none",
    remove_unused_columns=False,
    beta=BETA,
    max_length=MAX_SOURCE_LEN + MAX_TARGET_LEN, # 512
    gradient_checkpointing=True
)

# Tokenizer konfigurieren und explizit einstellen, dass er beim Tokenisieren die Längen kürzt
tokenizer.model_max_length = MAX_SOURCE_LEN + MAX_TARGET_LEN

# DPOTrainer initialisieren
# In der TRL-Dokumentation für Seq2Seq-Modelle (wie mBART) wird der Tokenizer über tokenizer=
# (oder processing_class=) übergeben. 
dpo_trainer = DPOTrainer(
    model=model, 
    ref_model=None, # Verwendet automatisch das Basismodell (ohne LoRA-Adapter) als Referenz!
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
    args=training_args
)

print("Starte DPO-Training mit Hugging Face...")
dpo_trainer.train()

Adding EOS to train dataset:   0%|          | 0/1241 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1241 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (759 > 512). Running this sequence through the model will result in indexing errors
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may 

Dropping fully truncated examples from train dataset:   0%|          | 0/1241 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/219 [00:00<?, ? examples/s]

...

[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized 

Dropping fully truncated examples from eval dataset:   0%|          | 0/219 [00:00<?, ? examples/s]

Starte DPO-Training mit Hugging Face...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
50,5.405600,2.164448,0.157875,735604.000000,27.635887,27.574133,0.002420,6.992438,6.536870,0.533333,0.455569,-3201.169710,-3013.085282
100,4.879400,2.181621,0.166025,1457109.000000,27.448795,27.083970,0.002726,11.476127,10.288353,0.533333,1.187774,-3156.332822,-2975.570447
150,5.749500,2.326943,0.162138,2191144.000000,27.249808,26.956068,0.002402,8.929589,8.031167,0.566667,0.898422,-3181.798201,-2998.142282
200,4.700800,2.497909,0.161542,2912237.000000,27.173242,26.759384,0.002544,9.010244,7.970611,0.588889,1.039633,-3180.991650,-2998.747827


TrainOutput(global_step=200, training_loss=5.721276206970215, metrics={'train_runtime': 359.0366, 'train_samples_per_second': 8.913, 'train_steps_per_second': 0.557, 'total_flos': 6704073061761024.0, 'train_loss': 5.721276206970215, 'epoch': 6.257028112449799})

In [14]:
# Trainiertes LoRA Modell speichern
dpo_trainer.save_model(os.path.join(OUTPUT_DIR, "best_dpo_lora"))
print("DPO LoRA Adapter erfolgreich gespeichert!")

DPO LoRA Adapter erfolgreich gespeichert!
